# Level 2: Linear Regression (SGD Edition)

## 🎯 Goal
Predict **Apartment Price** based on **Area** using **Gradient Descent**.

In the Streamlit app, you played the "Manual Fit Game". Here, we'll see how the computer plays that game automatically using **Stochastic Gradient Descent (SGD)**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Load Data
df = pd.read_csv('../data/seoul_apartment_price_2024.csv')
df = df.sample(n=1000, random_state=42) # Work with a sample for clarity
print(f"Loaded {len(df)} rows")

## 1. Visualizing the Challenge

We want to draw a line that fits this data.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['area_m2'], df['price_10k_krw'], alpha=0.3)
plt.title('Area vs Price')
plt.xlabel('Area (m²)')
plt.ylabel('Price (10k KRW)')
plt.show()

## 2. The Critical Step: Scaling

**Gradient Descent** is sensitive to the scale of data.
- Area: 0 ~ 200
- Price: 0 ~ 300,000

If we don't scale, the gradients will be crazy (huge updates for price, tiny for area). We use `StandardScaler` to make them look similar (mean=0, variance=1).

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X = df[['area_m2']].values
y = df[['price_10k_krw']].values

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y).ravel()

print("Scaled X range:", X_scaled.min(), X_scaled.max())
print("Scaled y range:", y_scaled.min(), y_scaled.max())

## 3. Training with SGD

Now we use `SGDRegressor`. This is the "Auto-Mixer" that adjusts weights step-by-step.

- **eta0 (Learning Rate)**: How big is the step?
- **max_iter (Epochs)**: How many times do we look at the data?

In [ ]:
# Try changing these!
LEARNING_RATE = 0.01
EPOCHS = 100

model = SGDRegressor(eta0=LEARNING_RATE, max_iter=EPOCHS, random_state=42)
model.fit(X_scaled, y_scaled)

print("Training Complete")

## 4. Evaluate & Visualize

We need to inverse transform the predictions to see the real price.

In [ ]:
# Predict on Scaled Data
y_pred_scaled = model.predict(X_scaled)

# Convert back to Real Price
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"RMSE: {rmse:,.0f} (10k KRW)")

# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(df['area_m2'], df['price_10k_krw'], alpha=0.3, label='Real Data')
plt.plot(df['area_m2'], y_pred, 'r-', linewidth=2, label='SGD Model')
plt.legend()
plt.title(f'Result (RMSE={rmse:,.0f})')
plt.xlabel('Area (m²)')
plt.ylabel('Price (10k KRW)')
plt.show()